# 02 · Data cleaning
**Project:** Sales Variance FP&A Dashboard

The cleaning is done by `scripts/clean_sales_data.py`, which reads `data/raw/sales_raw_messy_csv.xlsx` and writes `data/processed/Cleaned_Sales_Data.csv` and `.xlsx` (with an audit trail sheet).
This notebook **checks the result** against the raw file instead of re-running the script.

In [1]:
from pathlib import Path
import warnings
import pandas as pd

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_columns = 40

# Works whether Jupyter starts in notebooks/ or in the repo root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw" / "sales_raw_messy_csv.xlsx"
CLEAN = ROOT / "data" / "processed" / "Cleaned_Sales_Data.csv"
ENRICHED = ROOT / "data" / "processed" / "Sales_Variance_Enriched_Data.csv"
WORKBOOK = ROOT / "outputs" / "Sales_Variance_Analysis.xlsx"

## 1. Before vs after

In [2]:
raw = pd.read_excel(RAW, keep_default_na=False, na_values=[""])
clean = pd.read_csv(CLEAN, keep_default_na=False, na_values=[""], encoding="utf-8-sig")
print(f"Raw rows  : {len(raw):,}")
print(f"Clean rows: {len(clean):,}")
print(f"Removed   : {len(raw) - len(clean):,}  (exact and hidden duplicates)")

Raw rows  : 26,956
Clean rows: 25,920
Removed   : 1,036  (exact and hidden duplicates)


In [3]:
pd.DataFrame({
    "raw": raw["Region"].fillna("(blank)").value_counts(),
    "clean": clean["Region"].fillna("(blank)").value_counts(),
}).fillna(0).astype(int)

,raw,clean
Region,,
(blank),1163,0
APAC,1013,0
Asia Pacific,1005,5184
Asia Pacific,1024,0
Asia-Pacific,1031,0
AsiaPac,1082,0
EU,1326,0
Europa,1284,0
Europe,1301,5184


## 2. What was changed on each row
The `Cleaning_Notes` column records every repair; `Source_Excel_Row` points back to the row in the raw file.

In [4]:
notes = clean["Cleaning_Notes"].str.split("; ").explode().value_counts()
notes.to_frame("rows")

,rows
Cleaning_Notes,
Region standardised,19089
No changes needed,4022
Product case fixed,1981
Category trimmed,1579
Missing quantity rebuilt,1180
Gross margin recalculated,1174
Price rebuilt,1118
Blank Region recovered,1112
Cost rebuilt,1077


In [5]:
print(f"Rows with a repaired quantity: {clean['Cleaning_Notes'].str.contains('quantity', case=False).sum():,}")

Rows with a repaired quantity: 1,795


## 3. Integrity checks on the cleaned file

In [6]:
checks = {
    "5 standard regions only": clean["Region"].nunique() == 5,
    "No blank Region / Category / Product": clean[["Region", "Category", "Product"]].notna().all().all(),
    "No negative quantities": bool((clean["Actual_Qty"] >= 0).all() and (clean["Target_Qty"] >= 0).all()),
    "Source_Excel_Row is unique (every row traces to one raw row)": clean["Source_Excel_Row"].is_unique,
    "No exact duplicate rows": not clean.duplicated().any(),
}
pd.Series(checks, name="passed").to_frame()

,passed
5 standard regions only,True
No blank Region / Category / Product,True
No negative quantities,True
Source_Excel_Row is unique (every row traces to one raw row),True
No exact duplicate rows,True


## 4. Re-running the cleaning (optional)
From the repo root:
```
python scripts/clean_sales_data.py
```
It reads from `data/raw/` and rewrites the files in `data/processed/`.